# HomeBudget Domain Fine-Tuning Notebook

This notebook fine-tunes a small causal language model on the **HomeBudget** product/domain dataset.

The goal is to teach the model to answer accurately about:

- HomeBudget product behavior
- Budget wallets and category allocation
- Immutable ledger rules
- Currency and money precision rules
- Phase-wise product roadmap
- FastAPI/backend architecture
- Validation, security, testing, and offline-first rules

This notebook is designed to be clean, reproducible, and easy to update.

## Step 1 — Install dependencies

For Google Colab or a GPU runtime, the QLoRA setup needs `bitsandbytes`, `accelerate`, `peft`, `transformers`, and `datasets`.

In [ ]:
%pip install -U "transformers>=4.41.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.46.1" "trl>=0.8.6"

## Step 2 — Imports


In [ ]:
import os
import json
import torch

from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,TrainingArguments,Trainer,default_data_collator,set_seed
from peft import LoraConfig,TaskType,get_peft_model


## Step 3 — Configuration

Change these values when you switch model, dataset location, output directory, or training size.

Recommended starter model:

```text
TinyLlama/TinyLlama-1.1B-Chat-v1.0
```

It is small enough for experimentation and supports chat-style prompting.
## Some definations
seed = result repeatable banata hai

max_length = aik example ki max token length

epochs = dataset kitni dafa model dekhega

batch_size = aik time par kitne examples

gradient_accumulation = small GPU par fake larger batch

learning_rate = model kitni speed se seekhega

warmup_ratio = start mein slow learning

weight_decay = overfitting kam

logging_steps = logs kitni baar print hon

eval_steps = validation kitni baar ho

save_steps = checkpoint kitni baar save ho

lora_r = adapter ki learning capacity

lora_alpha = LoRA update ki strength

lora_dropout = overfitting protection




In [ ]:
CONFIG ={
    #Model
    "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",

    #Dataset files

    "train_file": "/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_train.jsonl" ,
    "validation_file": "/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_validation.jsonl",

     # Output
    "output_dir": "/mnt/data/homebudget_lora_adapter",
    "merged_output_dir": "/mnt/data/homebudget_merged_model",


     # Reproducibility

    "seed":42,

    #Tokenization

    "max_length":768,

    #Traning

    "num_train_epochs":3,
    "per_deveice_train_batch_size":1,
    "per_deveice_eval_batch_size":1,
    "gradient_accumulation_steps":8,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "logging_steps": 5,
    "eval_steps": 25,
    "save_steps": 25,

    #Lora
     "lora_r": 16,
     "lora_alpha": 32,
     "lora_dropout": 0.05,

}
set_seed(CONFIG["seed"])
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["merged_output_dir"]).mkdir(parents=True, exist_ok=True)

CONFIG

{'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'train_file': '/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_train.jsonl',
 'validation_file': '/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_validation.jsonl',
 'output_dir': '/mnt/data/homebudget_lora_adapter',
 'merged_output_dir': '/mnt/data/homebudget_merged_model',
 'seed': 42,
 'max_length': 768,
 'num_train_epochs': 3,
 'per_deveice_train_batch_size': 1,
 'per_deveice_eval_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'learning_rate': 0.0002,
 'warmup_ratio': 0.03,
 'weight_decay': 0.01,
 'logging_steps': 5,
 'eval_steps': 25,
 'save_steps': 25,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05}

## Step 4 — Load the raw dataset

The dataset uses a chat `messages` format:

```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."}
  ]
}

In [ ]:
data_files = {
    "train": CONFIG["train_file"],
    "validation": CONFIG["validation_file"],
}

dataset = load_dataset("json", data_files=data_files)
print(dataset)
print("Train Example")
print(json.dumps(dataset["train"][0],indent=2,ensure_ascii=False))
print("Validation Example")
print(json.dumps(dataset["validation"][0],indent=2,ensure_ascii=False))


DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'messages', 'source', 'format_version'],
        num_rows: 116
    })
    validation: Dataset({
        features: ['id', 'topic', 'messages', 'source', 'format_version'],
        num_rows: 20
    })
})
Train Example
{
  "id": "homebudget-0022",
  "topic": "budget",
  "messages": [
    {
      "role": "system",
      "content": "You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase."
    },
    {
      "role": "user",
      "content": "Can a user have multiple active budgets for the same period?"
    },
    {
      "role": "assistant",
      "content": "The current recommended rule says a user may have one active budget per cu

## Step 5 — Validate dataset structure

Before training, verify every row has exactly the fields expected by the formatter.

In [ ]:
def validate_record(record):
  if "messages" not in record:
    return False, "Missing messages field"

  messages = record["messages"]
  if not isinstance(messages,list) or len(messages)<3:
    return False,"messages must be a list with at least system/user/assistant"

  roles = [message.get("role") for message in messages]

  if "system" not in roles:
    return False,"Missing system message"
  if "user" not in roles:
    return False,"Missing user message"
  if "assistant" not in roles:
    return False,"Missing assistant message"

  for message in messages:
    if "content" not in message:
      return False,"Missing content field"

  return True,"ok"

for split in dataset:
  bad_rows=[]
  for index , row in enumerate(dataset[split]):
    ok,reason = validate_record(row)
    if not ok:
      bad_rows.append((index,reason))
  print(f"{split}: {len(dataset[split])} rows, invalid rows {len(bad_rows)}")

  if bad_rows[:5]:
    print(bad_rows[:5])



train: 116 rows, invalid rows 0
validation: 20 rows, invalid rows 0


## Step 6 — Load tokenizer

Important tokenizer rules:

- Set a pad token if the model does not already have one.
- Use the model's chat template when available.
- Keep the inference prompt format consistent with the training format.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"],use_fast = True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)
print("has_chat_template:", tokenizer.chat_template is not None)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pad_token: </s>
pad_token_id: 2
eos_token: </s>
eos_token_id: 2
has_chat_template: True


## Step 7 — Format prompts correctly

For supervised fine-tuning, the model should learn the **assistant answer**, not the system and user prompt.

This notebook masks prompt tokens with `-100`, so the loss is calculated only on the assistant response.

In [ ]:
def split_message(messages):
  """Return prompt messages and assistant response from a messages list."""
  assistant_message =[message for message in messages if message["role"] == "assistant"]
  if not assistant_message:
    raise ValueError("No assistant message found")

  assistant_text = assistant_message[-1]["content"].strip()

   # Keep everything before the final assistant answer as prompt context.

  prompt_message = []

  for message in messages:
    if message["role"]=="assistant":
      break
    prompt_message.append({
        "role":message["role"],
        "content":message["content"].strip()
    })

  return prompt_message,assistant_text

def render_prompt(prompt_message):
   """Render system/user messages into the same chat format used during inference."""
   if tokenizer.chat_template is not None:
    return tokenizer.apply_chat_template(prompt_message,tokenize=False,add_generation_prompt=True)

   # Fallback template for models without a tokenizer chat template.
   def render_prompt(prompt_message):
    """Render system/user messages into the same chat format used during inference."""
    if tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(
            prompt_message,
            tokenize=False,
            add_generation_prompt=True
        )

    text = ""
    for message in prompt_message:
        role = message["role"]
        content = message["content"]

        if role == "system":
            text += f"System: {content}\n"
        elif role == "user":
            text += f"User: {content}\n"
        else:
            text += f"{role.title()}: {content}\n"

    text += "Assistant:"
    return text





In [ ]:
def build_training_text(messages):
  prompt_message , assistent_text = split_message(messages)
  prompt_text = render_prompt(prompt_message)
  answer_text =  assistent_text + tokenizer.eos_token
  return prompt_text , answer_text

In [ ]:

sample_prompt, sample_answer = build_training_text(dataset["train"][0]["messages"])
print("PROMPT:")
print(sample_prompt)
print("ANSWER:")
print(sample_answer)

PROMPT:
<|system|>
You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase.</s>
<|user|>
Can a user have multiple active budgets for the same period?</s>
<|assistant|>

ANSWER:
The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change.</s>


## Step 8 — Tokenize with assistant-only labels

Bad approach:

```python
labels = input_ids.copy()
```

That trains the model to reproduce the system prompt and user question.

Better approach:

```python
labels = [-100 for prompt tokens] + [answer token ids]
```

That trains the model only on the assistant response.

In [ ]:
def tokinized_record(record):
  prompt_text,answer_text = build_training_text(record["messages"])
  prompt_ids = tokenizer(prompt_text,add_special_tokens=False)["input_ids"]
  answer_ids = tokenizer(answer_text,add_special_tokens=False)["input_ids"]
  input_ids = prompt_ids + answer_ids
  labels = [-100] * len(prompt_ids) + answer_ids
  attention_mask = [1]*len(input_ids)
  # Truncate from the right. For longer records, this may cut the answer.
  # If dataset has long answers, increase max_length in CONFIG.
  max_length = CONFIG["max_length"]
  input_ids = input_ids[:max_length]
  attention_mask = attention_mask[:max_length]
  labels = labels[:max_length]

  # Pad to max_length
  pad_length = max_length - len(input_ids)
  if pad_length > 0:
    input_ids += [tokenizer.pad_token_id] * pad_length
    labels += [-100] * pad_length
    attention_mask += [0] * pad_length

  return {
      "input_ids":input_ids,
      "labels":labels,
      "attention_mask":attention_mask
  }



In [ ]:
tokinzed = dataset.map(
      tokinized_record,
      remove_columns=dataset["train"].column_names,
  )

tokinzed

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 116
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 20
    })
})

## Step 9 — Check label masking

This sanity check confirms the prompt is masked and the assistant answer is trainable.

In [ ]:
def inspect_tokenized_example(index,split="train"):
  row = tokinzed[split][index]
  input_ids = row['input_ids']
  labels = row['labels']

  trainable_tokens = [token_ids for token_ids,lebel in zip(input_ids,labels) if lebel != -100]
  print("Full decoded sample:")
  print(tokenizer.decode(input_ids,skip_special_tokens=False))
  print("Trainable decoded target only:")
  print(tokenizer.decode(trainable_tokens,skip_special_tokens=False))
  print("Trainable tokens:", len(trainable_tokens))
  print("Total tokens:", sum(row["attention_mask"]))

inspect_tokenized_example(index,split="train")

Full decoded sample:
<|system|>
You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase.</s>
<|user|>
What belongs in the application layer?</s>
<|assistant|>
 The application layer contains use cases, services, schemas, and orchestration. It coordinates domain rules, repositories, validation, and database transactions to complete user operations safely.</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>

## Step 10 — Load model with QLoRA-ready quantization

This notebook uses 4-bit quantization for memory efficiency.

If your environment does not support `bitsandbytes`, switch to normal loading by removing `quantization_config`.

In [ ]:
from peft import prepare_model_for_kbit_training

In [ ]:
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype= compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

print("Model loaded:", CONFIG["model_name"])
print("Compute dtype:", compute_dtype)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Compute dtype: torch.bfloat16


## Step 11 — Add LoRA adapters

For Llama-style models, common target modules are:

```text
q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
```

This trains a small number of adapter weights instead of the full model.

In [ ]:
lora_config = LoraConfig(
    r = CONFIG["lora_r"],
    lora_alpha = CONFIG["lora_alpha"],
    lora_dropout = CONFIG["lora_dropout"],
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 12 — Training arguments

Notes:

- Use `eval_strategy` on newer Transformers versions.
- Some older versions use `evaluation_strategy`.
- If you get an argument error, replace `eval_strategy` with `evaluation_strategy`.

In [ ]:
traning_arguments = TrainingArguments(
    output_dir = CONFIG["output_dir"],
    num_train_epochs = CONFIG["num_train_epochs"],
    per_device_train_batch_size = CONFIG["per_deveice_train_batch_size"],
    per_device_eval_batch_size = CONFIG["per_deveice_eval_batch_size"],
    gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"],
    learning_rate = CONFIG["learning_rate"],
    warmup_steps = CONFIG["warmup_ratio"],
    weight_decay = CONFIG["weight_decay"],
    logging_steps = CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps = CONFIG["eval_steps"],
    save_steps = CONFIG["save_steps"],
    save_total_limit = 2,
    bf16 = (compute_dtype == torch.bfloat16),
    fp16 = (compute_dtype == torch.float16),
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
trainer = Trainer(
    model = model,
    args = traning_arguments,
    train_dataset = tokinzed["train"],
    eval_dataset = tokinzed["validation"],
    data_collator=default_data_collator,


)
train_result = trainer.train()
train_result

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,1.682482,1.740858
45,1.474115,1.676916


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=45, training_loss=1.8959903293185765, metrics={'train_runtime': 1148.0702, 'train_samples_per_second': 0.303, 'train_steps_per_second': 0.039, 'total_flos': 1679157809381376.0, 'train_loss': 1.8959903293185765, 'epoch': 3.0})

## Step 14 — Evaluate

In [ ]:
eval_result = trainer.evaluate()
eval_result

Training Loss,Validation Loss,Step
1.474115,1.676916,45


{'eval_loss': 1.6769163608551025}

## Step 15 — Save LoRA adapter and tokenizer

This saves the adapter, not a full merged model.

In [ ]:
trainer.model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Save LoRA adapter to {CONFIG["output_dir"]}")

Save LoRA adapter to /mnt/data/homebudget_lora_adapter


## Step 16 — Load adapter for inference

Correct way to load a PEFT/LoRA adapter:

1. Load the base model.
2. Attach the trained adapter with `PeftModel.from_pretrained`.
3. Use the same chat template as training.

In [ ]:
from peft import PeftModel

def load_model_for_inference():
  base_model = AutoModelForCausalLM.from_pretrained(
      CONFIG["model_name"],
      quantization_config =bnb_config,
      device_map="auto",
  )
  tuned_model = PeftModel.from_pretrained(
      base_model,
      CONFIG["output_dir"],

  )

  tuned_model.eval()

  return tuned_model

inference_model = load_model_for_inference()
print("Inference model ready")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Inference model ready


In [ ]:
from peft import PeftModel

def load_model_for_inference():
  base_model = AutoModelForCausalLM.from_pretrained(
      CONFIG["model_name"],
      quantization_config =bnb_config,
      device_map="auto",
  )
  tuned_model = PeftModel.from_pretrained(
      base_model,
      CONFIG["output_dir"],

  )

  tuned_model.eval()

  return tuned_model

inference_model = load_model_for_inference()
print("Inference model ready")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inference model ready


## Step 17 — Ask HomeBudget questions

In [ ]:
SYSTEM_PROMPT = (
    "You are HomeBudget Assistant. Answer questions about the HomeBudget app, "
    "its product rules, ledger design, architecture, APIs, validation, and implementation plan. "
    "Be accurate, practical, and grounded in the HomeBudget product requirements. "
    "Do not invent features that are outside the current phase; clearly say when something belongs to a later phase."
)

def build_chat_prompt(question):
  messages=[
      {'role':'system','content': SYSTEM_PROMPT},
      {'role':'user','content':question},
  ]

  if tokenizer.chat_template is not None:
    return tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
  return f"System: {SYSTEM_PROMPT} User:{question} Assestent: "

In [ ]:
def ask_homebudget(question,max_new_token=220):
  prompt = build_chat_prompt(question)
  inputs = tokenizer(prompt,return_tensors="pt").to(inference_model.device)

  with torch.no_grad():
    output_ids = inference_model.generate(
        **inputs,
        max_new_tokens = max_new_token,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_ids,skip_special_tokens=True).strip()
print(ask_homebudget("Explain HomeBudget in simple words."))

[transformers] Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


HomeBudget is a budgeting application that allows users to create budgets, allocate money from those budgets, track expenses, record transactions, and view balances. It does not include financial advice or transaction recommendations.


### Download the LoRA Adapter

In [ ]:
import shutil
from google.colab import files

output_dir = CONFIG["output_dir"]
archive_path = f"{output_dir}.zip"

# Create a zip archive of the output directory
shutil.make_archive(output_dir, 'zip', output_dir)

# Download the zip file
files.download(archive_path)

print(f"The LoRA adapter has been archived to '{archive_path}' and is ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The LoRA adapter has been archived to '/mnt/data/homebudget_lora_adapter.zip' and is ready for download.


# HomeBudget Domain Fine-Tuning Notebook

This notebook fine-tunes a small causal language model on the **HomeBudget** product/domain dataset.

The goal is to teach the model to answer accurately about:

- HomeBudget product behavior
- Budget wallets and category allocation
- Immutable ledger rules
- Currency and money precision rules
- Phase-wise product roadmap
- FastAPI/backend architecture
- Validation, security, testing, and offline-first rules

This notebook is designed to be clean, reproducible, and easy to update.

## Step 1 — Install dependencies

For Google Colab or a GPU runtime, the QLoRA setup needs `bitsandbytes`, `accelerate`, `peft`, `transformers`, and `datasets`.

In [ ]:
%pip install -U "transformers>=4.41.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.46.1" "trl>=0.8.6"

## Step 2 — Imports


In [ ]:
import os
import json
import torch

from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,TrainingArguments,Trainer,default_data_collator,set_seed
from peft import LoraConfig,TaskType,get_peft_model


## Step 3 — Configuration

Change these values when you switch model, dataset location, output directory, or training size.

Recommended starter model:

```text
TinyLlama/TinyLlama-1.1B-Chat-v1.0
```

It is small enough for experimentation and supports chat-style prompting.
## Some definations
seed = result repeatable banata hai

max_length = aik example ki max token length

epochs = dataset kitni dafa model dekhega

batch_size = aik time par kitne examples

gradient_accumulation = small GPU par fake larger batch

learning_rate = model kitni speed se seekhega

warmup_ratio = start mein slow learning

weight_decay = overfitting kam

logging_steps = logs kitni baar print hon

eval_steps = validation kitni baar ho

save_steps = checkpoint kitni baar save ho

lora_r = adapter ki learning capacity

lora_alpha = LoRA update ki strength

lora_dropout = overfitting protection




In [ ]:
CONFIG ={
    #Model
    "model_name": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",

    #Dataset files

    "train_file": "/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_train.jsonl" ,
    "validation_file": "/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_validation.jsonl",

     # Output
    "output_dir": "/mnt/data/homebudget_lora_adapter",
    "merged_output_dir": "/mnt/data/homebudget_merged_model",


     # Reproducibility

    "seed":42,

    #Tokenization

    "max_length":768,

    #Traning

    "num_train_epochs":3,
    "per_deveice_train_batch_size":1,
    "per_deveice_eval_batch_size":1,
    "gradient_accumulation_steps":8,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "logging_steps": 5,
    "eval_steps": 25,
    "save_steps": 25,

    #Lora
     "lora_r": 16,
     "lora_alpha": 32,
     "lora_dropout": 0.05,

}
set_seed(CONFIG["seed"])
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["merged_output_dir"]).mkdir(parents=True, exist_ok=True)

CONFIG

{'model_name': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'train_file': '/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_train.jsonl',
 'validation_file': '/content/drive/MyDrive/homebudget_sft_dataset/homebudget_messages_validation.jsonl',
 'output_dir': '/mnt/data/homebudget_lora_adapter',
 'merged_output_dir': '/mnt/data/homebudget_merged_model',
 'seed': 42,
 'max_length': 768,
 'num_train_epochs': 3,
 'per_deveice_train_batch_size': 1,
 'per_deveice_eval_batch_size': 1,
 'gradient_accumulation_steps': 8,
 'learning_rate': 0.0002,
 'warmup_ratio': 0.03,
 'weight_decay': 0.01,
 'logging_steps': 5,
 'eval_steps': 25,
 'save_steps': 25,
 'lora_r': 16,
 'lora_alpha': 32,
 'lora_dropout': 0.05}

## Step 4 — Load the raw dataset

The dataset uses a chat `messages` format:

```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."}
  ]
}

In [ ]:
data_files = {
    "train": CONFIG["train_file"],
    "validation": CONFIG["validation_file"],
}

dataset = load_dataset("json", data_files=data_files)
print(dataset)
print("Train Example")
print(json.dumps(dataset["train"][0],indent=2,ensure_ascii=False))
print("Validation Example")
print(json.dumps(dataset["validation"][0],indent=2,ensure_ascii=False))


DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'messages', 'source', 'format_version'],
        num_rows: 116
    })
    validation: Dataset({
        features: ['id', 'topic', 'messages', 'source', 'format_version'],
        num_rows: 20
    })
})
Train Example
{
  "id": "homebudget-0022",
  "topic": "budget",
  "messages": [
    {
      "role": "system",
      "content": "You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase."
    },
    {
      "role": "user",
      "content": "Can a user have multiple active budgets for the same period?"
    },
    {
      "role": "assistant",
      "content": "The current recommended rule says a user may have one active budget per cu

## Step 5 — Validate dataset structure

Before training, verify every row has exactly the fields expected by the formatter.

In [ ]:
def validate_record(record):
  if "messages" not in record:
    return False, "Missing messages field"

  messages = record["messages"]
  if not isinstance(messages,list) or len(messages)<3:
    return False,"messages must be a list with at least system/user/assistant"

  roles = [message.get("role") for message in messages]

  if "system" not in roles:
    return False,"Missing system message"
  if "user" not in roles:
    return False,"Missing user message"
  if "assistant" not in roles:
    return False,"Missing assistant message"

  for message in messages:
    if "content" not in message:
      return False,"Missing content field"

  return True,"ok"

for split in dataset:
  bad_rows=[]
  for index , row in enumerate(dataset[split]):
    ok,reason = validate_record(row)
    if not ok:
      bad_rows.append((index,reason))
  print(f"{split}: {len(dataset[split])} rows, invalid rows {len(bad_rows)}")

  if bad_rows[:5]:
    print(bad_rows[:5])



train: 116 rows, invalid rows 0
validation: 20 rows, invalid rows 0


## Step 6 — Load tokenizer

Important tokenizer rules:

- Set a pad token if the model does not already have one.
- Use the model's chat template when available.
- Keep the inference prompt format consistent with the training format.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"],use_fast = True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)
print("has_chat_template:", tokenizer.chat_template is not None)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pad_token: </s>
pad_token_id: 2
eos_token: </s>
eos_token_id: 2
has_chat_template: True


## Step 7 — Format prompts correctly

For supervised fine-tuning, the model should learn the **assistant answer**, not the system and user prompt.

This notebook masks prompt tokens with `-100`, so the loss is calculated only on the assistant response.

In [ ]:
def split_message(messages):
  """Return prompt messages and assistant response from a messages list."""
  assistant_message =[message for message in messages if message["role"] == "assistant"]
  if not assistant_message:
    raise ValueError("No assistant message found")

  assistant_text = assistant_message[-1]["content"].strip()

   # Keep everything before the final assistant answer as prompt context.

  prompt_message = []

  for message in messages:
    if message["role"]=="assistant":
      break
    prompt_message.append({
        "role":message["role"],
        "content":message["content"].strip()
    })

  return prompt_message,assistant_text

def render_prompt(prompt_message):
   """Render system/user messages into the same chat format used during inference."""
   if tokenizer.chat_template is not None:
    return tokenizer.apply_chat_template(prompt_message,tokenize=False,add_generation_prompt=True)

   # Fallback template for models without a tokenizer chat template.
   def render_prompt(prompt_message):
    """Render system/user messages into the same chat format used during inference."""
    if tokenizer.chat_template is not None:
        return tokenizer.apply_chat_template(
            prompt_message,
            tokenize=False,
            add_generation_prompt=True
        )

    text = ""
    for message in prompt_message:
        role = message["role"]
        content = message["content"]

        if role == "system":
            text += f"System: {content}\n"
        elif role == "user":
            text += f"User: {content}\n"
        else:
            text += f"{role.title()}: {content}\n"

    text += "Assistant:"
    return text





In [ ]:
def build_training_text(messages):
  prompt_message , assistent_text = split_message(messages)
  prompt_text = render_prompt(prompt_message)
  answer_text =  assistent_text + tokenizer.eos_token
  return prompt_text , answer_text

In [ ]:

sample_prompt, sample_answer = build_training_text(dataset["train"][0]["messages"])
print("PROMPT:")
print(sample_prompt)
print("ANSWER:")
print(sample_answer)

PROMPT:
<|system|>
You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase.</s>
<|user|>
Can a user have multiple active budgets for the same period?</s>
<|assistant|>

ANSWER:
The current recommended rule says a user may have one active budget per currency and overlapping period. This decision can be revisited later, but it should be enforced unless the product requirements change.</s>


## Step 8 — Tokenize with assistant-only labels

Bad approach:

```python
labels = input_ids.copy()
```

That trains the model to reproduce the system prompt and user question.

Better approach:

```python
labels = [-100 for prompt tokens] + [answer token ids]
```

That trains the model only on the assistant response.

In [ ]:
def tokinized_record(record):
  prompt_text,answer_text = build_training_text(record["messages"])
  prompt_ids = tokenizer(prompt_text,add_special_tokens=False)["input_ids"]
  answer_ids = tokenizer(answer_text,add_special_tokens=False)["input_ids"]
  input_ids = prompt_ids + answer_ids
  labels = [-100] * len(prompt_ids) + answer_ids
  attention_mask = [1]*len(input_ids)
  # Truncate from the right. For longer records, this may cut the answer.
  # If dataset has long answers, increase max_length in CONFIG.
  max_length = CONFIG["max_length"]
  input_ids = input_ids[:max_length]
  attention_mask = attention_mask[:max_length]
  labels = labels[:max_length]

  # Pad to max_length
  pad_length = max_length - len(input_ids)
  if pad_length > 0:
    input_ids += [tokenizer.pad_token_id] * pad_length
    labels += [-100] * pad_length
    attention_mask += [0] * pad_length

  return {
      "input_ids":input_ids,
      "labels":labels,
      "attention_mask":attention_mask
  }



In [ ]:
tokinzed = dataset.map(
      tokinized_record,
      remove_columns=dataset["train"].column_names,
  )

tokinzed

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 116
    })
    validation: Dataset({
        features: ['input_ids', 'labels', 'attention_mask'],
        num_rows: 20
    })
})

## Step 9 — Check label masking

This sanity check confirms the prompt is masked and the assistant answer is trainable.

In [ ]:
def inspect_tokenized_example(index,split="train"):
  row = tokinzed[split][index]
  input_ids = row['input_ids']
  labels = row['labels']

  trainable_tokens = [token_ids for token_ids,lebel in zip(input_ids,labels) if lebel != -100]
  print("Full decoded sample:")
  print(tokenizer.decode(input_ids,skip_special_tokens=False))
  print("Trainable decoded target only:")
  print(tokenizer.decode(trainable_tokens,skip_special_tokens=False))
  print("Trainable tokens:", len(trainable_tokens))
  print("Total tokens:", sum(row["attention_mask"]))

inspect_tokenized_example(index,split="train")

Full decoded sample:
<|system|>
You are HomeBudget Assistant. Answer questions about the HomeBudget app, its product rules, ledger design, architecture, APIs, validation, and implementation plan. Be accurate, practical, and grounded in the HomeBudget product requirements. Do not invent features that are outside the current phase; clearly say when something belongs to a later phase.</s>
<|user|>
What belongs in the application layer?</s>
<|assistant|>
 The application layer contains use cases, services, schemas, and orchestration. It coordinates domain rules, repositories, validation, and database transactions to complete user operations safely.</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s>

## Step 10 — Load model with QLoRA-ready quantization

This notebook uses 4-bit quantization for memory efficiency.

If your environment does not support `bitsandbytes`, switch to normal loading by removing `quantization_config`.

In [ ]:
from peft import prepare_model_for_kbit_training

In [ ]:
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype= compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

print("Model loaded:", CONFIG["model_name"])
print("Compute dtype:", compute_dtype)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Compute dtype: torch.bfloat16


## Step 11 — Add LoRA adapters

For Llama-style models, common target modules are:

```text
q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
```

This trains a small number of adapter weights instead of the full model.

In [ ]:
lora_config = LoraConfig(
    r = CONFIG["lora_r"],
    lora_alpha = CONFIG["lora_alpha"],
    lora_dropout = CONFIG["lora_dropout"],
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Step 12 — Training arguments

Notes:

- Use `eval_strategy` on newer Transformers versions.
- Some older versions use `evaluation_strategy`.
- If you get an argument error, replace `eval_strategy` with `evaluation_strategy`.

In [ ]:
traning_arguments = TrainingArguments(
    output_dir = CONFIG["output_dir"],
    num_train_epochs = CONFIG["num_train_epochs"],
    per_device_train_batch_size = CONFIG["per_deveice_train_batch_size"],
    per_device_eval_batch_size = CONFIG["per_deveice_eval_batch_size"],
    gradient_accumulation_steps = CONFIG["gradient_accumulation_steps"],
    learning_rate = CONFIG["learning_rate"],
    warmup_steps = CONFIG["warmup_ratio"],
    weight_decay = CONFIG["weight_decay"],
    logging_steps = CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps = CONFIG["eval_steps"],
    save_steps = CONFIG["save_steps"],
    save_total_limit = 2,
    bf16 = (compute_dtype == torch.bfloat16),
    fp16 = (compute_dtype == torch.float16),
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
trainer = Trainer(
    model = model,
    args = traning_arguments,
    train_dataset = tokinzed["train"],
    eval_dataset = tokinzed["validation"],
    data_collator=default_data_collator,


)
train_result = trainer.train()
train_result

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
25,1.682482,1.740858
45,1.474115,1.676916


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=45, training_loss=1.8959903293185765, metrics={'train_runtime': 1148.0702, 'train_samples_per_second': 0.303, 'train_steps_per_second': 0.039, 'total_flos': 1679157809381376.0, 'train_loss': 1.8959903293185765, 'epoch': 3.0})

## Step 14 — Evaluate

In [ ]:
eval_result = trainer.evaluate()
eval_result

Training Loss,Validation Loss,Step
1.474115,1.676916,45


{'eval_loss': 1.6769163608551025}

## Step 15 — Save LoRA adapter and tokenizer

This saves the adapter, not a full merged model.

In [ ]:
trainer.model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Save LoRA adapter to {CONFIG["output_dir"]}")

Save LoRA adapter to /mnt/data/homebudget_lora_adapter


## Step 16 — Load adapter for inference

Correct way to load a PEFT/LoRA adapter:

1. Load the base model.
2. Attach the trained adapter with `PeftModel.from_pretrained`.
3. Use the same chat template as training.

In [ ]:
from peft import PeftModel

def load_model_for_inference():
  base_model = AutoModelForCausalLM.from_pretrained(
      CONFIG["model_name"],
      quantization_config =bnb_config,
      device_map="auto",
  )
  tuned_model = PeftModel.from_pretrained(
      base_model,
      CONFIG["output_dir"],

  )

  tuned_model.eval()

  return tuned_model

inference_model = load_model_for_inference()
print("Inference model ready")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Inference model ready


In [ ]:
from peft import PeftModel

def load_model_for_inference():
  base_model = AutoModelForCausalLM.from_pretrained(
      CONFIG["model_name"],
      quantization_config =bnb_config,
      device_map="auto",
  )
  tuned_model = PeftModel.from_pretrained(
      base_model,
      CONFIG["output_dir"],

  )

  tuned_model.eval()

  return tuned_model

inference_model = load_model_for_inference()
print("Inference model ready")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Inference model ready


## Step 17 — Ask HomeBudget questions

In [ ]:
SYSTEM_PROMPT = (
    "You are HomeBudget Assistant. Answer questions about the HomeBudget app, "
    "its product rules, ledger design, architecture, APIs, validation, and implementation plan. "
    "Be accurate, practical, and grounded in the HomeBudget product requirements. "
    "Do not invent features that are outside the current phase; clearly say when something belongs to a later phase."
)

def build_chat_prompt(question):
  messages=[
      {'role':'system','content': SYSTEM_PROMPT},
      {'role':'user','content':question},
  ]

  if tokenizer.chat_template is not None:
    return tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
  return f"System: {SYSTEM_PROMPT} User:{question} Assestent: "

In [ ]:
def ask_homebudget(question,max_new_token=220):
  prompt = build_chat_prompt(question)
  inputs = tokenizer(prompt,return_tensors="pt").to(inference_model.device)

  with torch.no_grad():
    output_ids = inference_model.generate(
        **inputs,
        max_new_tokens = max_new_token,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(generated_ids,skip_special_tokens=True).strip()
print(ask_homebudget("Explain HomeBudget in simple words."))

[transformers] Both `max_new_tokens` (=220) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


HomeBudget is a budgeting application that allows users to create budgets, allocate money from those budgets, track expenses, record transactions, and view balances. It does not include financial advice or transaction recommendations.


### Download the LoRA Adapter

In [ ]:
import shutil
from google.colab import files

output_dir = CONFIG["output_dir"]
archive_path = f"{output_dir}.zip"

# Create a zip archive of the output directory
shutil.make_archive(output_dir, 'zip', output_dir)

# Download the zip file
files.download(archive_path)

print(f"The LoRA adapter has been archived to '{archive_path}' and is ready for download.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The LoRA adapter has been archived to '/mnt/data/homebudget_lora_adapter.zip' and is ready for download.
